# Suite EMB — Embedded objects and union values

A property whose schema is an `OfObject` holds an embedded object: a read-only record with no identity, written nested in
its owner's snapshot. A property whose schema is an `OfUnion` holds a value of one of its branches, recorded with the
branch it is written as (`{"$branch": index, "$value": value}` in snapshots). Validation checks union values against
their branches' predicates when given an evaluator; these cases use a stub for predicates written as `("has", name)`
and `("type", name)`.

In [ ]:
from mbse.Schemas.Framework import Comparison as C, JSON, Plain, Proxies, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError
from support import Fake, get_property, raises, set_property, stub_evaluator, text

Phone = S.OfObject.Builder().properties(text("number"), text("label")).create()
Email = S.OfObject.Builder().properties(text("address")).create()
Reach = S.OfUnion.Builder().branches(lambda b: b.of(Phone).when(("has", "number")),
                                     lambda b: b.of(Email).when(("has", "address"))).create()
Id = S.OfUnion.Builder().branches(lambda b: b.of(lambda t: t.as_native(int)).when(("type", "int")),
                                  lambda b: b.of(lambda t: t.as_native(str)).when(("type", "str"))).create()
Contact = S.OfObject.Builder().properties(
    text("name"), lambda p: p.name("home").of(Phone), lambda p: p.name("reach").of(Reach),
    lambda p: p.name("id").of(Id)).create()
Proxies.register("Contact", Contact)
B = Proxies.Builders


class Written:
    """A `Visitable` whose `accept` runs `write(visitor)`: data that neither proxies nor fakes produce."""

    def __init__(self, schema_name, write):
        self._schema_name, self._write = schema_name, write

    def identity(self):
        return id(self)

    def schema_name(self):
        return self._schema_name

    def accept(self, visitor):
        self._write(visitor)

## EMB-01 · An embedded object's schema has no adjacencies, however it is held

In [ ]:
assert Contact.validate() == [] and Reach.validate() == [] and Id.validate() == []
Owner = S.OfObject.Builder().properties(text("x")).create()
Links = S.OfRelation.Builder().links("a", "b").create()
S.OfObject.Builder(Owner).relations(lambda r: r.name("links").of(Links).me("a")).update()
Either = S.OfUnion.Builder().branches(lambda b: b.of(Owner).when(("has", "x")),
                                      lambda b: b.of(Phone).when(("has", "number"))).create()
Both = S.OfIntersection.Builder().of(Owner, Phone).create()
Holder = S.OfObject.Builder().properties(lambda p: p.name("owner").of(Owner), lambda p: p.name("either").of(Either),
                                         lambda p: p.name("both").of(Both)).create()
assert Holder.validate() == [
    "property 'owner': an embedded object cannot have adjacencies",
    "property 'either': an embedded object cannot have adjacencies",
    "property 'both': an embedded object cannot have adjacencies",
]
Pair = S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("inner").of(Owner)).create()
assert Pair.validate() == ["property 'inner': an embedded object cannot have adjacencies"]

## EMB-02 · Embedded objects: set with a Spec or a record, read, updated in part, cleared

In [ ]:
alice = B.Contact().name("Alice").home(lambda r: r.number("+44 1").label("home")).create()
assert alice.home.number == "+44 1" and alice.home.label == "home"
assert isinstance(alice.home, Proxies.OfObject.Record)
with raises(AttributeError, match="property 'label' is not set"):
    B.Contact().home(lambda r: r.number("1")).create().home.label
with raises(AttributeError, match="embedded objects are read-only; use a builder"):
    alice.home.number = "x"
with raises(AttributeError, match="nope"):
    alice.home.nope
assert B.Contact(alice).home(lambda r: r.label("work")).update() is alice  # starts from the value already set
assert alice.home.number == "+44 1" and alice.home.label == "work"
bob = B.Contact().name("Bob").home(alice.home).create()  # a record is a value, copied into the new owner
assert bob.home.number == "+44 1" and bob.home.label == "work"
B.Contact(bob).clear("home").update()
with raises(AttributeError, match="property 'home' is not set"):
    bob.home
assert alice.home.label == "work"
with raises(TypeError, match="property 'home' takes an embedded object or a Spec, got str"):
    B.Contact().home("+44")
with raises(AttributeError, match="'nope' is not a property of the embedded object"):
    B.Contact().home(lambda r: r.nope(1))

## EMB-03 · Union values: the value's kind chooses the branch, or `of` and `select` do

In [ ]:
carol = B.Contact().reach(lambda u: u.of(Email, lambda r: r.address("c@x"))).id(7).create()
assert carol.reach.address == "c@x" and carol.id == 7
dan = B.Contact().reach(carol.reach).id("D-1").create()  # the record's schema, the native's type
assert dan.reach.address == "c@x" and dan.id == "D-1"
with raises(TypeError, match="no branch of the union holds a float"):
    B.Contact().id(1.5)
with raises(TypeError, match="no branch of the union holds an embedded object"):
    B.Contact().id(carol.reach)
with raises(ValueError, match="no branch of the union has that schema"):
    B.Contact().reach(lambda u: u.of(Contact, lambda r: r))
with raises(ValueError, match="the union has no branch 5"):
    B.Contact().reach(lambda u: u.select(5))
with raises(ValueError, match="select a branch before writing the union's value"):
    B.Contact().reach(lambda u: u.value(lambda v: None))
seen = []
B.Contact(carol).property("reach", lambda p: p.value(lambda a: a.as_union(lambda u: seen.append(u.branch()))))
assert seen == [1]
with raises(AttributeError, match="property 'reach' is not set"):
    B.Contact().property("reach", lambda p: p.value(lambda a: a.as_union(lambda u: u.branch())))
B.Contact(carol).id(lambda u: u.value(lambda v: v.as_native(lambda n: n.clear()))).update()  # clearing the value
with raises(AttributeError, match="property 'id' is not set"):
    carol.id
B.Contact(carol).reach(lambda u: u.select(0).value(lambda v: v.as_object(lambda r: r.number("9")))).update()
assert carol.reach.number == "9"  # another branch: its value starts empty
with raises(TypeError, match="property 'name' does not hold an object"):
    B.Contact().property("name", lambda p: p.value(lambda a: a.as_object(lambda o: None)))
with raises(TypeError, match="property 'name' does not hold a union"):
    B.Contact().property("name", lambda p: p.value(lambda a: a.as_union(lambda u: None)))

## EMB-04 · Snapshots: embedded objects nested, union values with their branch; strict decoding

In [ ]:
eve = (B.Contact().name("Eve").home(lambda r: r.number("1"))
       .reach(lambda u: u.of(Phone, lambda r: r.number("2").label("m"))).id("E").create())
expected = JSON.ToJSON(Contact, eve)
assert expected == ('{"root": "s0", "objects": {"s0": {"name": "Eve", "home": {"number": "1"}, "reach": {"$branch": 0, '
                    '"$value": {"number": "2", "label": "m"}}, "id": {"$branch": 1, "$value": "E"}}}}')
for encode, decode in [(Plain.ToPlain, Plain.FromPlain), (JSON.ToJSON, JSON.FromJSON), (YAML.ToYAML, YAML.FromYAML)]:
    assert JSON.ToJSON(Contact, decode(B)(Contact, encode(Contact, eve))) == expected

FromPlain = Plain.FromPlain(B)
for s0, message in [
    ({"home": 3}, "$.objects.s0.home: an embedded object must be a mapping, got int"),
    ({"home": {"x": "1"}}, "$.objects.s0.home.x: the embedded object has no property 'x'"),
    ({"home": {"number": 1}}, "$.objects.s0.home.number: expected str, got int"),
    ({"reach": {"number": "1"}}, "$.objects.s0.reach: a union value is {'$branch': index, '$value': value}"),
    ({"id": {"$branch": 2, "$value": 1}}, "$.objects.s0.id.$branch: the union has no branch 2"),
    ({"id": {"$branch": True, "$value": 1}}, "$.objects.s0.id.$branch: the union has no branch True"),
    ({"id": {"$branch": 0, "$value": "x"}}, "$.objects.s0.id.$value: expected int, got str"),
]:
    with raises(DecodeError, match=message):
        FromPlain(Contact, {"root": "s0", "objects": {"s0": s0}})
with raises(TypeError, match="property 'home' is not native"):
    Plain.ToPlain(Contact, Fake("Contact", {"home": "x"}))

Box = S.OfObject.Builder().properties(text("x")).create()
Slot = S.OfRelation.Builder().links("box", "item").properties(lambda p: p.name("tag").of(Phone)).create()
S.OfObject.Builder(Box).relations(lambda r: r.name("items").of(Slot).me("box"),
                                  lambda r: r.name("boxes").of(Slot).me("item")).update()
Proxies.register("Box", Box)
Proxies.register("Slot", Slot)
with raises(NotImplementedError, match="entry property 'tag': entry properties must be native"):
    FromPlain(Box, {"root": "s0", "objects": {"s0": {"items": [{"item": {"$ref": "s0", "$schema": "Box"}, "tag": "x"}]}}})

## EMB-05 · Validation: embedded values recursively, and union branches with an evaluator

In [ ]:
validate, checked = Validators.Validate(B), Validators.Validate(B, stub_evaluator)
assert validate(Contact, eve) == [] and checked(Contact, eve) == []
odd = B.Contact().reach(lambda u: u.of(Phone, lambda r: r.label("no number"))).create()
assert validate(Contact, odd) == []  # without an evaluator, only kinds and types are checked
assert checked(Contact, odd) == ["Contact#0.reach: no branch's predicate holds for the value"]
Twice = S.OfUnion.Builder().branches(lambda b: b.of(lambda t: t.as_native(str)).when(("type", "str")),
                                     lambda b: b.of(lambda t: t.as_native(str)).when(("type", "str"))).create()
Note = S.OfObject.Builder().properties(lambda p: p.name("text").of(Twice)).create()
Proxies.register("Note", Note)
note = B.Note().text(lambda u: u.select(1).value(lambda v: v.as_native(lambda n: n.set("t")))).create()
assert checked(Note, note) == ["Note#0.text: written as branch 1, but the first branch whose predicate holds is 0"]
assert validate(Contact, Fake("Contact", {"home": "x", "reach": 3})) == [
    "Contact#0.home: expected an embedded object, got str", "Contact#0.reach: expected a union value, got int"]
assert validate(Contact, B.Contact().home(lambda r: r.number(5)).create()) == ["Contact#0.home.number: expected str, got int"]
stray = Written("Contact", lambda v: v.property("home", lambda p: p.value(lambda a: a.as_object(
    lambda o: set_property(o, "fax", "1")))))
assert validate(Contact, stray) == ["Contact#0.home.fax: not a property of the embedded object"]
nine = Written("Contact", lambda v: v.property("id", lambda p: p.value(lambda a: a.as_union(
    lambda u: u.select(9).value(lambda w: w.as_native(lambda n: n.set(1)))))))
assert validate(Contact, nine) == ["Contact#0.id: the union has no branch 9"]
assert validate(Contact, Written("Contact", lambda v: v.property("id", lambda p: p.value(lambda a: a.as_object(
    lambda o: None))))) == ["Contact#0.id: expected a union value, got an embedded object"]
Both = S.OfIntersection.Builder().of(Phone, Email).create()
Mixed = S.OfObject.Builder().properties(lambda p: p.name("both").of(Both)).create()
Proxies.register("Mixed", Mixed)
assert validate(Mixed, Fake("Mixed", {"both": 1})) == ["Mixed#0.both: intersection values cannot be validated yet"]
with raises(NotImplementedError, match="intersection values are not supported by Plain yet"):
    FromPlain(Mixed, {"root": "s0", "objects": {"s0": {"both": 1}}})
assert validate(Contact, Written("Contact", lambda v: v.property("name", lambda p: p.value(lambda a: a.as_union(
    lambda u: u.select(0).value(lambda w: w.as_native(lambda n: n.set(1)))))))) == [
    "Contact#0.name: expected str, got a union value"]
box = Fake("Box")
box.adjacencies["items"] = [{"item": box, "tag": "x"}]
assert validate(Box, box) == ["Box#0.items[0].tag: entry properties must be native"]

values = Validators.properties_of(eve)
assert values["id"] == "E"  # a union value reads as the value it holds
assert Validators.properties_of(values["home"]) == {"number": "1"}
assert Validators.properties_of(values["reach"]) == {"number": "2", "label": "m"}
Tagged = S.OfObject.Builder().properties(lambda p: p.name("id").of(Id)).create()
Sleeve = S.OfObject.Builder().properties(lambda p: p.name("tagged").of(Tagged)).create()
Proxies.register("Sleeve", Sleeve)
sleeve = B.Sleeve().tagged(lambda r: r.id(5)).create()
assert Validators.properties_of(Validators.properties_of(sleeve)["tagged"]) == {"id": 5}

## EMB-06 · Comparison: embedded objects by their properties, union values by branch and value

In [ ]:
def record(value):
    return C.OfObject(Contact, value)


assert record(eve).compare(record(B.Contact(eve).clone())) == 0
assert record(eve).compare(record(B.Contact(eve).home(lambda r: r.label("x")).clone())) is None
assert record(eve).compare(record(B.Contact(eve).id(3).clone())) is None  # another branch
assert record(eve).compare(record(B.Contact(eve).id("F").clone())) is None
assert record(B.Contact().create()).compare(record(B.Contact().create())) == 0
union = C.OfUnion(Id)
with raises(ValueError, match="no branch is selected"):
    union.branch()
with raises(ValueError, match="the union has no branch 2"):
    union.select(2)
with raises(ValueError, match="the union has no branch 1.5"):
    union.select(1.5)
assert union.compare(C.OfUnion(Id).select(0).value(lambda v: v.as_native(lambda n: n.set(1)))) is None  # one is absent
union.select(0).value(lambda v: v.as_native(lambda n: n.set(1)))
assert union.select(0).branch() == 0 and union.compare(C.OfUnion(Reach)) is None  # reselecting keeps the value
other = C.OfUnion(Id).select(0).value(lambda v: v.as_native(lambda n: n.set(1)))
assert union.compare(other) == 0 and other.compare(C.OfUnion(Id)) is None
assert C.OfUnion(Id).compare(C.OfUnion(Id)) == 0  # both are absent
emptied = C.OfAny(Id).as_union(lambda u: u.select(0).value(lambda v: v.as_native(lambda n: n.set(1).clear())))
assert emptied.compare(C.OfAny(Id)) == 0  # a union whose value was cleared is absent
word = C.OfAny(S.OfNative.Data(str)).as_native(lambda n: n.set("b"))
assert C.OfAny(Id).as_union(lambda u: u.select(1).value(lambda v: v.as_native(lambda n: n.set("b")))).compare(word) is None
twice = C.OfAny(Id)
twice.as_union(lambda u: u.select(1).value(lambda v: v.as_native(lambda n: n.set("a"))))
twice.as_union(lambda u: u.value(lambda v: v.as_native(lambda n: n.set("b"))))
assert twice.compare(C.OfAny(Id).as_union(lambda u: u.select(1).value(lambda v: v.as_native(lambda n: n.set("b"))))) == 0

## EMB-07 · The new visitor classes read back through their protocols

In [ ]:
# Proxies: an embedded object's builder, read through Visitors.OfObject.
seen = []
B.Contact(alice).property("home", lambda p: p.value(lambda a: a.as_object(lambda r: (
    r.properties(lambda q: seen.append(q.name())), seen.append(r.has("label")), r.clear("label"),
    seen.append(r.has("label")), r.adjacencies(seen.append)))))
assert seen == ["number", "label", True, False]
with raises(AttributeError, match="an embedded object has no adjacencies, got 'x'"):
    B.Contact().home(lambda r: r.adjacency("x", lambda a: None))
with raises(AttributeError, match="'nope' is not a property of the embedded object"):
    B.Contact().home(lambda r: r.property("nope", lambda q: None))
B.Contact().home(lambda r: seen.append(hasattr(r, "_private")))  # Python's own probes are not properties
assert seen[-1] is False

# Plain: writers for embedded objects and unions, and their errors.
out = {}
writer = Plain._ObjectWriter(out, Contact, lambda t: {})
writer.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "number", "1"))))
writer.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "label", "h"))))
assert out == {"home": {"number": "1", "label": "h"}}  # writing again adds to the same mapping
with raises(TypeError, match="property 'name' is not an object"):
    writer.property("name", lambda p: p.value(lambda a: a.as_object(lambda o: None)))
with raises(TypeError, match="property 'name' is not a union"):
    writer.property("name", lambda p: p.value(lambda a: a.as_union(lambda u: None)))
writer.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: (
    seen.append(u.select(1).branch()), u.value(lambda v: v.as_native(lambda n: n.set("I")))))))
assert out["id"] == {"$branch": 1, "$value": "I"}
writer.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: u.select(1).value(lambda v: None))))
assert "id" not in out  # nothing written: no value
with raises(ValueError, match="no branch is selected"):
    writer.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: u.branch())))
with raises(ValueError, match="the union has no branch 3"):
    writer.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: u.select(3))))

# Validators: the recorder for union values.
recorder = Validators._ObjectRecord()
recorder.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: u.select(0).value(
    lambda v: v.as_native(lambda n: n.set(1))))))
recorder.property("id", lambda p: p.value(lambda a: a.as_union(lambda u: seen.append(u.branch()))))
recorder.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "number", "1"))))
recorder.property("home", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "label", "h"))))
assert Validators.properties_of(Validators.properties_of(recorder)["home"]) == {"number": "1", "label": "h"}
assert seen[-1] == 0
with raises(ValueError, match="no branch is selected"):
    Validators._UnionRecord().branch()